# shard-infer — một shard của infer.py hybrid (kaggle_orchestra)

Bản shard của `infer_hybrid.ipynb`. `params.adapter_loc` là chuỗi phải có trong đường
dẫn adapter (vd `phien5`). Mốc chỉ ghi khi đủ câu của khoảng — hết `--gio-pha2` giữa
chừng thì orchestrator đổi acc và nối tiếp từ `prompts.jsonl` + `du_doan.jsonl`.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 4, "commit": "", "noi_tiep": None, "output": "x.jsonl"}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

# Codebase chép ra /tmp chứ KHÔNG ra /kaggle/working: mọi thứ trong working là output, và
# orchestrator tải output của từng shard về máy dev sau mỗi lần dừng — 30 MB repo × số lần.
SRC = next((c.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
# Mọi shard phải chạy CÙNG commit: dataset src của từng acc được đẩy riêng, lệch một acc
# là ghép hai phiên bản code vào một tập mà không dấu vết.
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
# Nối tiếp: dataset `orch-<job>-s<i>` mang phần đã làm từ acc trước.
if SHARD["noi_tiep"]:
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print(f"nối tiếp {f.name}: {sum(1 for _ in open(OUT / f.name, encoding='utf-8')):,} dòng")

In [ ]:
!pip -q install transformers==5.14.1 peft nltk
!pip -q uninstall -y torchao

In [ ]:
os.environ["LEGALQA_SO_HIEU"] = "1"
from legalqa.prompt import CO_SO_HIEU
from build_sft import TOP_N, MAX_CHARS
assert CO_SO_HIEU is True and TOP_N == 5 and MAX_CHARS > 10**6

ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
CLEAN = Path("/tmp/clean"); CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    if not (CLEAN / f.name).exists():
        (CLEAN / f.name).symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)

CAU_HOI = next(Path("/kaggle/input").glob("**/public-official.json"), None)
assert CAU_HOI, "Không thấy public-official.json — gắn dataset official"
_d = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert len(_d) == 1000, f"tập test có {len(_d)} câu, phải là 1.000"

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/tmp/base"); FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX

ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")
                if SHARD["adapter_loc"] in str(q)), None)
assert ADAPTER, f"không thấy tot_nhat/ có '{SHARD['adapter_loc']}' trong đường dẫn"
local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    RERANK = str(local[0])
else:
    from huggingface_hub import snapshot_download
    RERANK = snapshot_download("AITeamVN/Vietnamese_Reranker")
from huggingface_hub import snapshot_download
DENSE = snapshot_download("intfloat/multilingual-e5-small")
print("base", BASE, "| adapter", ADAPTER, "| rerank", RERANK, "| dense", DENSE)

In [ ]:
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI), "--tu", str(SHARD["tu"]), "--den", str(SHARD["den"]),
    "--model", str(BASE), "--adapter", str(ADAPTER), "--rerank", RERANK, "--dense", DENSE,
    "--ra", str(OUT), "--batch-size", "32", "--pool", "50",
    "--max-new", str(SHARD["max_new"]), "--gen-bs", str(SHARD["gen_bs"]),
    "--gio-pha1", str(SHARD["gio_pha1"]), "--gio-pha2", str(SHARD["gio_pha2"]),
], check=True)